### Importing Libraries

In [37]:
import pandas as pd
import random

### Random Data Creation

In [38]:
data = []
for _ in range(50000):
    order_value = random.randint(500, 50000)
    city_tier = random.choice(['Rural', 'Urban', 'Metro City'])
    payment_type = random.choice(['COD', 'COD', 'Credit Card', 'Debit Card', 'UPI', 'Net Banking'])
    return_count = random.randint(0,10)
    is_verified_mobile = random.choice([True, False])

    is_rto = 0
    if payment_type == "COD":
        if return_count > 3:
            is_rto = random.choices([1,0], weights = [90,10])[0]
        elif city_tier == 'Rural' and not is_verified_mobile:
            is_rto = random.choices([1,0], weights = [70,30])[0]
        else:
            is_rto = random.choices([1,0], weights=[30,70])[0]
    else:
        is_rto = random.choices([1,0], weights=[2,98]) [0]
    data.append([order_value, city_tier, payment_type, return_count, is_verified_mobile, is_rto])


df = pd.DataFrame(data, columns=['Order_Value', 'City_Tier', 'Payment_Type', 'Return_Count', 'Is_Verified_Mobile', 'Is_RTO'])
df.to_csv('ecommerce_rto_data.csv', index=False)


### Data Loading

In [39]:
df = pd.read_csv('ecommerce_rto_data.csv')

In [40]:
df.head()

,Order_Value,City_Tier,Payment_Type,Return_Count,Is_Verified_Mobile,Is_RTO
0,47320,Rural,COD,3,False,0
1,9737,Urban,Net Banking,9,True,0
2,17461,Metro City,Debit Card,9,False,0
3,3576,Urban,COD,10,False,1
4,4815,Metro City,Net Banking,10,False,0


### 1. Seperating the data columns for question and answer.

In [41]:
X =df.drop('Is_RTO', axis=1)
y = df['Is_RTO']

### 2. Train test split data

In [42]:
from sklearn.model_selection import train_test_split

In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
import sklearn
sklearn.set_config(enable_metadata_routing = True)

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

### 3. One hot encoding data.

In [45]:
categorical_column = ['City_Tier','Payment_Type']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_column)
    ], remainder='passthrough')

### 4. Model Pipeline

In [46]:
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42))
])

### 5. Model Traning & Testing

In [47]:
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

TypeError: All intermediate steps should be transformers and implement fit and transform or be the string 'passthrough' 'SMOTE(random_state=42)' (type <class 'imblearn.over_sampling._smote.base.SMOTE'>) doesn't